# Deep Model LOSO Evaluation — Colab
Run full 23-fold Leave-One-Subject-Out for FusionModel across all 3 targets.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    GITHUB_REPO = "https://github.com/YOUR_USERNAME/emotion-recognition.git"
    !git clone {GITHUB_REPO} /content/emotion-recognition
    %cd /content/emotion-recognition
    !pip install mat73 pyyaml -q

    from google.colab import drive
    drive.mount("/content/drive")

    import shutil, os
    os.makedirs("data/raw", exist_ok=True)
    shutil.copy("/content/drive/MyDrive/DREAMER.mat", "data/raw/DREAMER.mat")
    print("✅ Ready")

In [ ]:
import sys, json, os
sys.path.insert(0, ".")

from src.utils.config import load_config
from src.training.cross_subject_eval import run_loso

config = load_config("configs/default.yaml")

In [ ]:
# ── Choose model ──────────────────────────────────────────────────
MODEL_TYPE = "fusion"    # eegnet | cnn | cnnlstm | fusion
all_summaries = {}

for target in ["valence", "arousal", "dominance"]:
    print(f"\n{'#'*60}")
    print(f"  Running LOSO for: {target.upper()}")
    print(f"{'#'*60}")

    summary = run_loso(
        mat_path       = config["data"]["raw_path"],
        target         = target,
        model_type     = MODEL_TYPE,
        config         = config,
        save_dir       = "outputs/results",
        checkpoint_dir = "outputs/models",
    )
    all_summaries[target] = summary

In [ ]:
import pandas as pd

rows = []
for target, s in all_summaries.items():
    rows.append({
        "target"  : target,
        "model"   : s["model"],
        "folds"   : s["n_folds"],
        "acc"     : f"{s['acc_mean']} ± {s['acc_std']}",
        "f1"      : f"{s['f1_mean']}  ± {s['f1_std']}",
        "auc"     : f"{s['auc_mean']} ± {s['auc_std']}",
    })

df = pd.DataFrame(rows)
print("\n── LOSO Deep Model Results ──")
print(df.to_string(index=False))
df.to_csv("outputs/results/deep_loso_summary.csv", index=False)

In [ ]:
if IN_COLAB:
    !git config --global user.email "you@example.com"
    !git config --global user.name "Your Name"
    !git add outputs/results/ outputs/models/ notebooks/
    !git commit -m "feat: deep LOSO results for all targets"
    !git push
    print("✅ Pushed to GitHub")